In [224]:
import os,sys
sys.path.insert(1, os.path.join(os.getcwd()  , '..'))

In [225]:
import importlib, shallowsim as sb
import pandas as pd
import math

In [226]:
importlib.reload(sb)

<module 'shallowsim' from '/home/boyingchen/shallowsim/shallowsim.py'>

In [227]:
args = sb.ModelArgs.load_from_csv("modelArgs.csv","qwen3")
c = sb.Config()

In [228]:
print(args.is_moe)
print(args.attention)

True
gqa


In [229]:
gpu_blackwell = sb.get_gpu_info('./device/gpu_info.csv',print_console=True ) 

| gpu_type      |   sm |   comm_sm |   fp16 |   fp8 |   fp4 |   mem |   mem_bw |   nvlink_bw |   pcie_bw |   gpu_per_node |
|:--------------|-----:|----------:|-------:|------:|------:|------:|---------:|------------:|----------:|---------------:|
| DGX-B300      |  160 |        20 |   3375 |  7500 | 15000 |   288 |     8000 |         900 |       100 |              8 |
| DGX-B200      |  160 |        20 |   2250 |  4500 |  9000 |   180 |     8000 |         900 |       100 |              8 |
| GB200-NVL72   |  160 |        20 |   2500 |  5000 | 10000 |   192 |     8000 |         900 |       100 |             72 |
| GB300-NVL72   |  160 |        20 |   3750 |  7500 | 15000 |   288 |     8000 |         900 |       100 |             72 |
| Rubin-NVL144  |  110 |        12 |   6400 | 12800 | 25600 |   144 |     6500 |         900 |       100 |            144 |
| RubinU-NVL576 |  110 |        12 |   6500 | 13000 | 26000 |   256 |     8000 |        1350 |       100 |            576 |
| H200  

In [230]:
seq_len = 4383
kv_cache_rate = 0.563
decode_len = 1210
bs_list =[ 16, 32, 64, 128, 256, 512]
eplist = [ 8 , 16, 36, 72, 144, 320]

In [231]:
detail,summary = sb.prefill_time(args,gpu_blackwell,seq_len, kv_cache_rate, tp=4, dp=8)

In [232]:
detail

GPU,Layers,DGX-B300,DGX-B200,GB200-NVL72,GB300-NVL72,Rubin-NVL144,RubinU-NVL576,H200,H800,H20,H20-3E,MI300X,MI308X
GQA,0.0,0.249747,0.376320,0.338927,0.226750,0.132058,0.129519,0.949149,0.949947,6.152317,6.151519,0.676938,4.635373
DenseMLP,0.0,0.139820,0.218916,0.199142,0.139820,0.094340,0.088403,0.962668,0.969727,6.076820,6.069761,0.692905,4.578540
TP-GQA,94.0,0.172198,0.203841,0.194493,0.166448,0.142775,0.127220,0.391809,0.503911,1.692601,1.692402,0.324156,1.313765
Shared Expert,94.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Combine,94.0,0.352137,0.352137,0.138123,0.138123,0.138123,0.108749,0.654274,0.654274,0.654274,0.654274,0.654274,0.654274
Overlap1,94.0,0.179939,0.148296,-0.056370,-0.028325,-0.004652,-0.018472,0.262465,0.150363,-1.038327,-1.038128,0.330118,-0.659491
Routed Expert,94.0,0.034955,0.054729,0.049786,0.034955,0.023585,0.022101,0.240667,0.242432,1.519205,1.517440,0.173226,1.144635
Dispatch,94.0,0.125534,0.125534,0.072031,0.072031,0.072031,0.064687,0.201068,0.201068,0.201068,0.201068,0.201068,0.201068
Overlap2,94.0,0.090579,0.070805,0.022245,0.037076,0.048446,0.042587,-0.039599,-0.041363,-1.318137,-1.316372,0.027842,-0.943566


In [223]:
summary

GPU,DGX-B300,DGX-B200,GB200-NVL72,GB300-NVL72,Rubin-NVL144,RubinU-NVL576,H200,H800,H20,H20-3E,MI300X,MI308X
Compute,19.472371,24.305586,22.962173,18.931932,15.637876,14.036175,59.452782,70.156262,301.909791,301.725153,46.753949,231.089594
Comm,44.901091,44.901091,19.754485,19.754485,19.754485,16.302990,80.402183,80.402183,80.402183,80.402183,80.402183,80.402183
Sum,44.901091,44.901091,25.053223,22.417049,20.191786,18.039311,84.124454,84.290337,301.909791,301.725153,80.402183,231.089594


In [197]:
detail,summary = sb.prefill_time_pp(args, c, gpu_blackwell, kv_cache_rate, tp=4, dp=8,bs=32, pp=4, micro_bs = 1) # support tp_list /dp_list or dp in Config c

In [198]:
summary
# serial total: Sum * bs

GPU,DGX-B300,DGX-B200,GB200-NVL72,GB300-NVL72,Rubin-NVL144,RubinU-NVL576,H200,H800,H20,H20-3E,MI300X,MI308X
Compute,44.007206,61.221441,56.269710,41.414515,29.584204,27.024238,163.455698,176.097174,958.458037,957.174671,121.434578,725.355665
Comm,44.133625,44.133625,16.980641,16.980641,17.069376,13.342496,82.467251,82.467251,82.467251,82.467251,82.467251,82.467251
Sum,49.838061,62.232083,56.269710,41.414515,29.584204,27.024238,163.455698,176.097174,958.458037,957.174671,121.434578,725.355665
Bubble,78.433669,97.939015,88.555609,65.176942,46.558747,42.529949,257.241754,277.136537,1508.392976,1506.373253,191.110155,1141.543341
Total_PP,431.385181,538.664585,487.055848,358.473182,256.073111,233.914720,1414.829646,1524.250953,8296.161367,8285.052891,1051.105854,6278.488376
Serial_Total,1594.817943,1991.426648,1800.630710,1325.264493,946.694532,864.775632,5230.582328,5635.109583,30670.657173,30629.589477,3885.906490,23211.381269
Speedup,3.696970,3.696970,3.696970,3.696970,3.696970,3.696970,3.696970,3.696970,3.696970,3.696970,3.696970,3.696970


In [199]:
tp = 4
_, ttft_sum = sb.prefill_time(args, gpu_blackwell, seq_len, kv_cache_rate, tp=tp, dp=8, print_console=False)
print(ttft_sum.apply(lambda x: seq_len/tp * (1000/ x)).loc['Sum'].to_markdown(floatfmt=".1f"))

| GPU           |     Sum |
|:--------------|--------:|
| DGX-B300      | 21986.2 |
| DGX-B200      | 17607.5 |
| GB200-NVL72   | 19473.2 |
| GB300-NVL72   | 26458.1 |
| Rubin-NVL144  | 37038.3 |
| RubinU-NVL576 | 40546.9 |
| H200          |  6703.7 |
| H800          |  6222.4 |
| H20           |  1143.2 |
| H20-3E        |  1144.8 |
| MI300X        |  9023.4 |
| MI308X        |  1510.6 |


# Test decode (time & PP)

In [200]:
args = sb.ModelArgs.load_from_csv("modelArgs.csv","deepseek-v3")
c = sb.Config()

In [201]:
print(args.is_moe)
print(args.attention)

True
mla


In [202]:
gpu_blackwell_decode = sb.get_gpu_info('./device/gpu_info.csv', decoding_mode=True, print_console=True) 

| gpu_type      |   sm |   comm_sm |   fp16 |   fp8 |   fp4 |   mem |   mem_bw |   nvlink_bw |   pcie_bw |   gpu_per_node |
|:--------------|-----:|----------:|-------:|------:|------:|------:|---------:|------------:|----------:|---------------:|
| DGX-B300      |  160 |        20 |   3375 |  7500 | 15000 |   288 |     8000 |         900 |       100 |              8 |
| DGX-B200      |  160 |        20 |   2250 |  4500 |  9000 |   180 |     8000 |         900 |       100 |              8 |
| GB200-NVL72   |  160 |        20 |   2500 |  5000 | 10000 |   192 |     8000 |         900 |       100 |             72 |
| GB300-NVL72   |  160 |        20 |   3750 |  7500 | 15000 |   288 |     8000 |         900 |       100 |             72 |
| Rubin-NVL144  |  110 |        12 |   6400 | 12800 | 25600 |   144 |     6500 |         900 |       100 |            144 |
| RubinU-NVL576 |  110 |        12 |   6500 | 13000 | 26000 |   256 |     8000 |        1350 |       100 |            576 |
| H200  

In [203]:

detail = sb.decode_time(
        args,                     
        gpu_blackwell_decode,                
        c.bs_list,           
        c.seq_len,
        c.decode_len,
        gemm_group_per_device=math.ceil(args.n_routed_experts / 8),  
        device_num=8,             
        fp8_combine=False,
        tps_limit=0,
        print_console=True) 


|              |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B200 |   DGX-B200 |   DGX-B200 |   DGX-B200 |   DGX-B200 |   DGX-B200 |   DGX-B200 |   DGX-B200 |   DGX-B200 |   DGX-B200 |   GB200-NVL72 |   GB200-NVL72 |   GB200-NVL72 |   GB200-NVL72 |   GB200-NVL72 |   GB200-NVL72 |   GB200-NVL72 |   GB200-NVL72 |   GB200-NVL72 |   GB200-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   Rubin-NVL144 |   Rubin-NVL144 |   Rubin-NVL144 |   Rubin-NVL144 |   Rubin-NVL144 |   Rubin-NVL144 |   Rubin-NVL144 |   RubinU-NVL576 |   RubinU-NVL576 |   RubinU-NVL576 |   RubinU-NVL576 |   RubinU-NVL576 |   RubinU-NVL576 |   RubinU-NVL576 |   RubinU-NVL576 |   RubinU-NVL576 |   RubinU-NVL576 |   RubinU-NVL576 |

In [204]:
result = sb.decode_time_pp(
        args,
        c,                                          
        gpu_blackwell_decode,            
        gemm_group_per_device=32,  
        device_num=8,   
        pp=4,micro_bs=1,                   # pipeline parallelism          
        fp8_combine=False,
        tps_limit=0,
        print_console=True) 


[Decode · Pipeline-parallel]
|              |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B300 |   DGX-B200 |   DGX-B200 |   DGX-B200 |   DGX-B200 |   DGX-B200 |   DGX-B200 |   DGX-B200 |   DGX-B200 |   DGX-B200 |   DGX-B200 |   GB200-NVL72 |   GB200-NVL72 |   GB200-NVL72 |   GB200-NVL72 |   GB200-NVL72 |   GB200-NVL72 |   GB200-NVL72 |   GB200-NVL72 |   GB200-NVL72 |   GB200-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   GB300-NVL72 |   Rubin-NVL144 |   Rubin-NVL144 |   Rubin-NVL144 |   Rubin-NVL144 |   Rubin-NVL144 |   Rubin-NVL144 |   Rubin-NVL144 |   RubinU-NVL576 |   RubinU-NVL576 |   RubinU-NVL576 |   RubinU-NVL576 |   RubinU-NVL576 |   RubinU-NVL576 |   RubinU-NVL576 |   RubinU-NVL576 |   RubinU-NVL576 |   Rub

In [ ]:
dfs_o = detail.groupby(['GPU','BatchSize'],as_index=False).apply(lambda t: t[t.Total==t.Total.max()]).sort_values(['Total'],ascending=False).reset_index(drop=True)

dfs_o.style.bar(subset=['TPS','Total'],color='#6495ED')\
      .applymap(sb.gpu_category_color,props=sb.gpu_category_idx(gpu_blackwell_decode),subset=['GPU'])\
      .format(precision=3) 

/tmp/ipykernel_3826198/2282476199.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dfs_o = detail.groupby(['GPU','BatchSize'],as_index=False).apply(lambda t: t[t.Total==t.Total.max()]).sort_values(['Total'],ascending=False).reset_index(drop=True)
/tmp/ipykernel_3826198/2282476199.py:4: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(sb.gpu_category_color,props=sb.gpu_category_idx(gpu_blackwell_decode),subset=['GPU'])\


,GPU,TP,BatchSize,DenseGQA,DenseMLP,SparseGQA,Combine,SharedExpert,RoutedExpert,Dispatch,COMP_SUM,COMM_SUM,Delta,TPOT,TPOT_O,TPS,TPS_O,Total,Total_O,Comm_Impact
0,RubinU-NVL576,1,256,0.185,0.115,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,24.011,24.011,41.647,41.647,10661.607,10661.607,0.000
1,GB300-NVL72,1,256,0.198,0.127,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,26.027,26.027,38.421,38.421,9835.867,9835.867,0.000
2,DGX-B300,1,256,0.201,0.127,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,26.275,26.275,38.059,38.059,9743.053,9743.053,0.000
3,RubinU-NVL576,1,128,0.097,0.107,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,16.342,16.342,61.190,61.190,7832.358,7832.358,0.000
4,GB300-NVL72,1,128,0.104,0.113,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,17.350,17.350,57.636,57.636,7377.368,7377.368,0.000
5,DGX-B300,1,128,0.105,0.113,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,17.474,17.474,57.227,57.227,7325.031,7325.031,0.000
6,GB200-NVL72,1,128,0.112,0.120,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,18.542,18.542,53.933,53.933,6903.426,6903.426,0.000
7,DGX-B200,1,128,0.114,0.122,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,18.939,18.939,52.802,52.802,6758.694,6758.694,0.000
8,Rubin-NVL144,1,128,0.118,0.130,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,19.818,19.818,50.459,50.459,6458.742,6458.742,0.000
9,RubinU-NVL576,1,64,0.053,0.103,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,12.508,12.508,79.949,79.949,5116.727,5116.727,0.000


In [ ]:
sb.df_filter(detail,'DGX-B300',0).style\
      .bar(subset=['TPS','Total'],color='#6495ED')\
      .applymap(sb.color_positive_red, subset=['Delta'])\
      .background_gradient(subset=['Comm_Impact'],cmap=sb.cm)\
      .format(precision=3) 

/tmp/ipykernel_3826198/995444697.py:3: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(sb.color_positive_red, subset=['Delta'])\


,GPU,TP,BatchSize,DenseGQA,DenseMLP,SparseGQA,Combine,SharedExpert,RoutedExpert,Dispatch,COMP_SUM,COMM_SUM,Delta,TPOT,TPOT_O,TPS,TPS_O,Total,Total_O,Comm_Impact
0,DGX-B300,1,16,0.022,0.101,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,9.774,9.774,102.316,102.316,1637.058,1637.058,0.000
1,DGX-B300,4,16,0.022,0.101,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,9.774,9.774,102.316,102.316,1637.058,1637.058,0.000
2,DGX-B300,8,16,0.022,0.101,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,9.774,9.774,102.316,102.316,1637.058,1637.058,0.000
3,DGX-B300,1,32,0.034,0.102,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,10.874,10.874,91.965,91.965,2942.873,2942.873,0.000
4,DGX-B300,4,32,0.034,0.102,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,10.874,10.874,91.965,91.965,2942.873,2942.873,0.000
5,DGX-B300,8,32,0.034,0.102,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,10.874,10.874,91.965,91.965,2942.873,2942.873,0.000
6,DGX-B300,1,64,0.058,0.106,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,13.074,13.074,76.488,76.488,4895.238,4895.238,0.000
7,DGX-B300,4,64,0.058,0.106,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,13.074,13.074,76.488,76.488,4895.238,4895.238,0.000
8,DGX-B300,1,128,0.105,0.113,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,17.474,17.474,57.227,57.227,7325.031,7325.031,0.000
9,DGX-B300,1,256,0.201,0.127,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,26.275,26.275,38.059,38.059,9743.053,9743.053,0.000


In [ ]:
args = sb.ModelArgs.load_from_csv("modelArgs.csv","qwen3")
c = sb.Config()
c.bs_list=[16,32,64,128,256,512,1024,2048]
gpu_blackwell_decode = sb.get_gpu_info('./device/gpu_info.csv', 
                                    decoding_mode=True, print_console=True) 

| gpu_type      |   sm |   comm_sm |   fp16 |   fp8 |   fp4 |   mem |   mem_bw |   nvlink_bw |   pcie_bw |   gpu_per_node |
|:--------------|-----:|----------:|-------:|------:|------:|------:|---------:|------------:|----------:|---------------:|
| DGX-B300      |  160 |        20 |   3375 |  7500 | 15000 |   288 |     8000 |         900 |       100 |              8 |
| DGX-B200      |  160 |        20 |   2250 |  4500 |  9000 |   180 |     8000 |         900 |       100 |              8 |
| GB200-NVL72   |  160 |        20 |   2500 |  5000 | 10000 |   192 |     8000 |         900 |       100 |             72 |
| GB300-NVL72   |  160 |        20 |   3750 |  7500 | 15000 |   288 |     8000 |         900 |       100 |             72 |
| Rubin-NVL144  |  110 |        12 |   6400 | 12800 | 25600 |   144 |     6500 |         900 |       100 |            144 |
| RubinU-NVL576 |  110 |        12 |   6500 | 13000 | 26000 |   256 |     8000 |        1350 |       100 |            576 |
| H200  

In [ ]:
dfs = sb.decode_time_with_ep_list(args,gpu_blackwell_decode,c,print_console=False,fp8_combine=True,tps_limit=0)

In [ ]:
dfs_o = dfs.groupby(['GPU','BatchSize'],as_index=False).apply(lambda t: t[t.Total==t.Total.max()]).sort_values(['Total'],ascending=False).reset_index(drop=True)
dfs_o.style.bar(subset=['TPS','Total'],color='#6495ED')\
      .applymap(sb.gpu_category_color,props=sb.gpu_category_idx(gpu_blackwell_decode),subset=['GPU'])\
      .applymap(sb.color_positive_red, subset=['Delta'])\
      .background_gradient(subset=['Comm_Impact'],cmap=sb.cm)\
      .format(precision=3) 


/tmp/ipykernel_3826198/3398479797.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  dfs_o = dfs.groupby(['GPU','BatchSize'],as_index=False).apply(lambda t: t[t.Total==t.Total.max()]).sort_values(['Total'],ascending=False).reset_index(drop=True)
/tmp/ipykernel_3826198/3398479797.py:3: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(sb.gpu_category_color,props=sb.gpu_category_idx(gpu_blackwell_decode),subset=['GPU'])\
/tmp/ipykernel_3826198/3398479797.py:4: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(sb.color_positive_red, subset=['Delta'])\


,GPU,TP,EP,BatchSize,DenseGQA,DenseMLP,SparseGQA,Combine,SharedExpert,RoutedExpert,Dispatch,COMP_SUM,COMM_SUM,Delta,TPOT,TPOT_O,TPS,TPS_O,Total,Total_O,Comm_Impact
0,RubinU-NVL576,1,144,512,0.094,0.028,0.144,0.024,0.004,0.013,0.024,0.161,0.048,-0.100,15.114,15.114,66.162,66.162,33875.092,33875.092,0.000
1,RubinU-NVL576,1,72,512,0.094,0.028,0.144,0.024,0.004,0.013,0.024,0.161,0.048,-0.100,15.114,15.114,66.162,66.162,33875.092,33875.092,0.000
2,RubinU-NVL576,1,320,512,0.094,0.028,0.144,0.024,0.004,0.013,0.024,0.161,0.048,-0.100,15.114,15.114,66.162,66.162,33875.092,33875.092,0.000
3,GB300-NVL72,1,72,512,0.104,0.033,0.154,0.031,0.005,0.020,0.031,0.179,0.062,-0.097,16.798,16.798,59.532,59.532,30480.229,30480.229,0.000
4,GB200-NVL72,1,72,512,0.115,0.039,0.165,0.031,0.006,0.029,0.031,0.200,0.062,-0.109,18.787,18.787,53.228,53.228,27252.488,27252.488,0.000
5,DGX-B300,1,8,512,0.106,0.033,0.156,0.031,0.005,0.041,0.031,0.203,0.062,-0.100,19.066,19.066,52.450,52.450,26854.313,26854.313,0.000
6,RubinU-NVL576,1,144,256,0.048,0.025,0.098,0.017,0.003,0.008,0.017,0.109,0.034,-0.068,10.269,10.269,97.385,97.385,24930.538,24930.538,0.000
7,RubinU-NVL576,1,320,256,0.048,0.025,0.098,0.017,0.003,0.008,0.017,0.109,0.034,-0.068,10.269,10.269,97.385,97.385,24930.538,24930.538,0.000
8,RubinU-NVL576,1,72,256,0.048,0.025,0.098,0.017,0.003,0.008,0.017,0.109,0.034,-0.068,10.269,10.269,97.385,97.385,24930.538,24930.538,0.000
9,GB300-NVL72,1,72,256,0.053,0.027,0.103,0.020,0.004,0.011,0.020,0.118,0.041,-0.066,11.110,11.110,90.007,90.007,23041.801,23041.801,0.000


In [ ]:
sb.df_filter(dfs,'Rubin-NVL144',0).style\
      .bar(subset=['TPS','Total'],color='#6495ED')\
      .applymap(sb.color_positive_red, subset=['Delta'])\
      .background_gradient(subset=['Comm_Impact'],cmap=sb.cm)\
      .format(precision=3) 

/tmp/ipykernel_3826198/4080135161.py:3: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(sb.color_positive_red, subset=['Delta'])\


,GPU,TP,EP,BatchSize,DenseGQA,DenseMLP,SparseGQA,Combine,SharedExpert,RoutedExpert,Dispatch,COMP_SUM,COMM_SUM,Delta,TPOT,TPOT_O,TPS,TPS_O,Total,Total_O,Comm_Impact
49,Rubin-NVL144,1,8,16,0.006,0.026,0.056,0.011,0.004,0.029,0.011,0.089,0.021,-0.039,8.351,8.351,119.753,119.753,1916.053,1916.053,0.000
50,Rubin-NVL144,4,8,16,0.006,0.026,0.069,0.011,0.004,0.029,0.011,0.101,0.021,-0.051,9.539,9.539,104.837,104.837,1677.391,1677.391,0.000
51,Rubin-NVL144,8,8,16,0.006,0.026,0.069,0.011,0.004,0.029,0.011,0.101,0.021,-0.051,9.499,9.499,105.273,105.273,1684.373,1684.373,0.000
52,Rubin-NVL144,1,8,32,0.010,0.027,0.060,0.011,0.004,0.031,0.011,0.094,0.023,-0.041,8.850,8.850,112.989,112.989,3615.659,3615.659,0.000
53,Rubin-NVL144,4,8,32,0.010,0.027,0.072,0.011,0.004,0.031,0.011,0.107,0.023,-0.054,10.024,10.024,99.757,99.757,3192.214,3192.214,0.000
54,Rubin-NVL144,8,8,32,0.010,0.027,0.072,0.011,0.004,0.031,0.011,0.106,0.023,-0.053,9.980,9.980,100.201,100.201,3206.433,3206.433,0.000
55,Rubin-NVL144,1,8,64,0.017,0.027,0.067,0.013,0.004,0.031,0.013,0.101,0.025,-0.045,9.498,9.498,105.286,105.286,6738.301,6738.301,0.000
56,Rubin-NVL144,4,8,64,0.017,0.027,0.079,0.013,0.004,0.031,0.013,0.113,0.025,-0.057,10.644,10.644,93.952,93.952,6012.950,6012.950,0.000
57,Rubin-NVL144,1,8,128,0.031,0.028,0.081,0.015,0.004,0.031,0.015,0.116,0.030,-0.054,10.860,10.860,92.080,92.080,11786.221,11786.221,0.000
58,Rubin-NVL144,1,8,256,0.058,0.030,0.108,0.020,0.004,0.035,0.020,0.147,0.041,-0.072,13.853,13.853,72.187,72.187,18479.893,18479.893,0.000
